# Generalization Deep Dive: Uncertainty, Reach, and Power

**Topic 10 · 2 lectures**

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/honr_464_logo.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb10_inference_student.ipynb)

---

## 🧭 Inquiry & Claim Boundary

**Inquiry emphasis:** the **generalization** position, taken slowly. Generalization
means carrying an answer from the people you actually measured to a larger group
you did not measure. Example: you survey 400 undergraduates and want to speak
about all undergraduates at the university. In the book's compass this is not a
separate kind of inquiry. It is a **descriptive** question whose **estimand**
(the exact quantity the question asks for, such as "the average trust score
across everyone") lives in a population (RDSS ch. 7). The crossing is *earned*
by a sampling data strategy (ch. 8), *carried* by an answer strategy that
reports its uncertainty (ch. 9), and *proven* by diagnosis, meaning coverage and
power checks (ch. 10). This topic walks that whole move in two lectures: how
much your estimate wobbles and how far it honestly reaches (Lecture 1), then
whether your design could have detected anything at all, and why "moves
together" never quietly becomes "causes" (Lecture 2).

| | |
|---|---|
| **A claim this topic PERMITS** | "My estimate is X; under my design, estimates this far from the truth are rare. Here is the interval, and here is the power that produced it." |
| **A claim this topic does NOT permit** | A bare point estimate narrated as the truth ("the effect IS 2.0"); a sample narrated as its population with no sampling design or uncertainty behind it (**the silent upgrade**, the compass's first unlicensed crossing); an association quietly promoted to "impact" or "effect"; or "we found nothing, so there is nothing" from a design that could barely have found anything. |

**Where this sits in the course:** Phase 3, data and answer strategies. It
develops milestones M06 (your Answer Strategy declaration) and M07 (the URC
abstract plus the full one-page design declaration). Lecture 1 lands right
before the abstract's internal gate at this week's Friday studio, so its
uncertainty language goes straight into your draft. Lecture 2 lands right
before the studio where the full declaration is assembled. Everything builds on
the world you declared in nb04 and the estimators you hand-rolled in nb09.

*Provenance: RDSS ch. 7 (SATE vs PATE reach) + ch. 8 (sampling data strategy) + ch. 9–10 (uncertainty, diagnosing designs: coverage/power) + ch. 15–16 boundary + rdss-problem-set-1 (power) | generalization as a descriptive inquiry with a population estimand; uncertainty & power via simulation | sampling-distribution + power-grid simulations built on the source concepts | newly-constructed-from-source-concept*

## Learning Objectives

By the end of this notebook, you will be able to:

1. Build a design's **sampling distribution** by simulation, and read its
   **standard error** off the spread.
2. Explain a **confidence interval** as a property of a *procedure* run many
   times, not a fact about your one estimate.
3. Use the **√n rule** to reason about how much precision more data actually buys.
4. Name who your result honestly speaks for, and show how **non-response** can
   tilt an answer in a way no bigger sample can fix.
5. Estimate a design's **statistical power** by simulation and locate the single
   change that would raise it most.
6. Keep your claims on the association side of the association-causation
   boundary, and carry uncertainty-first and power language into your URC
   abstract and full design declaration (M07).

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

> 💡 **Gemini Prompt:** "This Python builds a simulated world where each person has TWO belonging scores: Y0 (no mentor) and Y1 (with a mentor), with Y1 = Y0 + effect. Explain why a real study can never observe both Y0 and Y1 for the same person, and why that missing half is the reason statistical inference from data exists at all: [paste the next cell]."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Confirm in the printed output that the one-study estimate lands near the true effect of 2 but not exactly on it, and be able to say why a new random assignment would move it.
> - [ ] Check that the code really sets Y1 = Y0 + effect, so the true average effect is exactly the number passed in, not something the estimate has to discover.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# The mentoring-program world returns — the same world you declared in nb04.
# Each person has TWO potential belonging scores (0-100): Y0 without a mentor,
# Y1 with one. By construction Y1 = Y0 + effect, so the true average effect is
# EXACTLY the `effect` you pass in. We never see both columns for one person in
# real life — that is the whole problem this deep dive exists to manage.

def make_world(n=100, effect=2.0, noise=2.0, rng=rng):
    """n people, each with a no-mentor score Y0 and a with-mentor score Y1."""
    Y0 = rng.normal(50, 10, n) + rng.normal(0, noise, n)
    return pd.DataFrame({"Y0": Y0.round(1), "Y1": (Y0 + effect).round(1)})

def complete_ra(n, m, rng):
    """Randomly assign exactly m of n people to the mentor (Z=1), the rest to Z=0."""
    z = np.zeros(n, dtype=int); z[:m] = 1
    return rng.permutation(z)

def difference_in_means(df, y="Y", z="Z"):
    """Our answer strategy from nb09: the treated mean minus the control mean,
    plus the standard error of that difference."""
    g1, g0 = df.loc[df[z] == 1, y], df.loc[df[z] == 0, y]
    est = g1.mean() - g0.mean()
    se = np.sqrt(g1.var(ddof=1) / len(g1) + g0.var(ddof=1) / len(g0))
    return est, se

TRUE_ATE = 2.0  # the genie's answer — true by construction

# Run ONE study: declare a 100-person world, assign, reveal, estimate.
demo = make_world(n=100, rng=np.random.default_rng(SEED))
demo["Z"] = complete_ra(100, 50, np.random.default_rng(SEED))
demo["Y"] = np.where(demo["Z"] == 1, demo["Y1"], demo["Y0"])
est, se = difference_in_means(demo)
print(f"✓ One study, n=100: estimate = {est:.2f}  (truth = {TRUE_ATE})")
print(f"  Run it again with a different random assignment and this number MOVES.")

# Lecture 1

## 1. Why This Matters

> *"Show me your point estimate and I'll show you a number that would have come
> out differently if you had run the study on a Tuesday. I don't want your
> number. I want to know how much it would have jumped around, and whether you
> know."*
> — a **thesis advisor** reading a first draft, kindly refusing to be impressed

You just computed one estimate of the mentoring effect. It landed near 2, but
not on it. Re-run the whole study with the same design and a new random
assignment, and you get a *different* number. Not because the truth changed.
Because *which half of the people got a mentor* changed. That wobble has a
name: **sampling variability**, the movement in a result that comes purely from
who happened to land in your sample or your groups. Example: poll 100 random
people twice and the two polls disagree by a point or two, even though opinion
never moved.

The wobble is not a mistake in your analysis. It is the very thing statistical
inference measures, and refusing to report it is how honest researchers
accidentally mislead.

Here is the move this lecture teaches. Instead of running the study once and
hoping, you run it **1,000 times** inside the computer and watch where the
estimates pile up. The spread of that pile tells you how uncertain any single
answer is. That spread also has a name, the **standard error**: the typical
distance between estimates like yours and the truth. Example: an estimate of
2.1 with a standard error of 1 says studies built like yours usually miss the
truth by about a point. The standard error is the raw material of every
interval and every honest abstract you will ever write.

The whole sample-to-population move looks like this:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/population_sample_inference.png" width="70%"/></center>

*From a population you cannot fully measure, you take one sample and then infer: a single point estimate, plus an interval estimate that carries its own confidence. (From Professor Moreira's QM 67000 Business Analytics slides.)*

> **A question that often comes up here:** *"If I only get to run my real study
> once, why does it help to imagine running it a thousand times?"* Because the
> thousand imagined runs tell you how much your one real run could have missed
> by luck of the draw. You are not pretending to have more data. You are
> measuring how trustworthy the data you have will be. That measure is what
> lets you write "roughly 2, give or take" instead of the overconfident "2.0",
> and it is exactly the language your URC abstract needs when it meets its
> internal gate at this week's studio.

---

### 🔮 Pause & Predict

We are about to run the mentoring study 1,000 times at **n = 100**, then 1,000
times again at **n = 400**, four times the people. The pile of 1,000 estimates
will have some spread at each size. **Before running**, commit a prediction in
the cell below. Going from n = 100 to n = 400 is a 4× increase in sample size.
Will the spread of the estimates be **cut to one-quarter**, **cut in half**, or
**stay about the same**? One sentence on why.

### YOUR ANSWER HERE:

**Going from n = 100 to n = 400, the spread will be (¼ / ½ / same):**

**Why:**

---

### 🛠️ Run the Study

The cell below packages the four steps you already know (declare a world,
assign, reveal, estimate) into one function that does them **1,000 times at
once** using fast array math. Nothing new is happening; it is the same study on
repeat. Read the result table, then compare it against your prediction.

> 💡 **Gemini Prompt:** "This code runs the same two-group study 1,000 times at n = 100 and again at n = 400, then reports the spread of the 1,000 estimates at each size: [paste the next cell]. Before I run it, predict the ratio of the n = 400 spread to the n = 100 spread, and explain why the √n rule says quadrupling the sample should roughly HALVE the spread rather than quarter it."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Compare Gemini's predicted spread ratio against the printed "Spread ratio n400 / n100". Is it close to the 0.50 the √n rule implies?
> - [ ] Check that the "spread of estimates" and the "average reported SE" columns land near each other at each size; if Gemini claims otherwise, trust the table.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# The engine: run the whole design `reps` times and collect every estimate.
# It is the four steps from the cell above, vectorized so 1,000 runs take a blink.
def run_design(n, effect=2.0, noise=2.0, reps=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    Y0 = rng.normal(50, 10, (reps, n)) + rng.normal(0, noise, (reps, n))
    Y1 = Y0 + effect
    m = n // 2
    treated = np.argsort(rng.random((reps, n)), axis=1) < m   # exactly m treated per run
    Y = np.where(treated, Y1, Y0)
    mean1 = (Y * treated).sum(1) / m
    mean0 = (Y * ~treated).sum(1) / (n - m)
    var1 = ((Y - mean1[:, None]) ** 2 * treated).sum(1) / (m - 1)
    var0 = ((Y - mean0[:, None]) ** 2 * ~treated).sum(1) / (n - m - 1)
    est = mean1 - mean0
    se = np.sqrt(var1 / m + var0 / (n - m))
    return pd.DataFrame({"est": est, "se": se})

small = run_design(n=100, reps=1000)
large = run_design(n=400, reps=1000)

summary = pd.DataFrame({
    "spread of estimates (the SE)": [small.est.std(ddof=1), large.est.std(ddof=1)],
    "average estimate":            [small.est.mean(),       large.est.mean()],
    "average reported SE":         [small.se.mean(),        large.se.mean()],
}, index=["n = 100", "n = 400"]).round(3)
print(summary.to_string())
print(f"\nTrue effect (by construction): {TRUE_ATE}")
print(f"Spread ratio n400 / n100: {large.est.std(ddof=1) / small.est.std(ddof=1):.3f}"
      f"  (0.50 would mean 'exactly halved')")

In [ ]:
# The picture: the histogram of 1,000 estimates IS your uncertainty.
fig, ax = plt.subplots()
bins = np.linspace(-2, 6, 41)
ax.hist(small.est, bins=bins, alpha=0.6, label="n = 100", color="#c44")
ax.hist(large.est, bins=bins, alpha=0.6, label="n = 400", color="#268")
ax.axvline(TRUE_ATE, color="k", linestyle="--", label=f"truth = {TRUE_ATE}")
ax.set_xlabel("estimated mentoring effect (one bar = many simulated studies)")
ax.set_ylabel("how many of the 1,000 studies landed here")
ax.set_title("Same design, 1,000 runs: the wider the pile, the less sure any one answer is")
ax.legend()
plt.show()

# Self-check: 4x the sample size should roughly HALVE the spread (the √n rule),
# and both piles should center on the truth.
ratio = large.est.std(ddof=1) / small.est.std(ddof=1)
assert 0.42 < ratio < 0.58, f"spread ratio {ratio:.3f} — expected ≈ 0.5 (the √n rule)"
assert abs(small.est.mean() - TRUE_ATE) < 0.2, "n=100 estimates should center on truth"
print(f"✓ Self-check passed: 4× the data cut the spread to {ratio:.2f} of its size (≈ ½).")

The pile you just drew has a name and a shape: a **sampling distribution**, the
full collection of estimates a design produces when you run it over and over.
The textbook this course draws on plots the very same thing:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_9_1.png" width="65%"/></center>

*Run one design many times and its estimates spread into a bell around the truth; the estimand (the value the design aims at) and the mean of the simulated estimates land almost on the same line. (Figure from the replication materials of Blair, Coppock & Humphreys (2023), *Research Design in the Social Sciences* (MIT-licensed archive; the book is free at book.declaredesign.org).)*

One more picture, because it fixes the single most misread idea in all of
statistics: what "95% confident" actually means. A **confidence interval** is a
range built around your estimate by a fixed recipe, wide enough that the recipe
catches the true value most of the time. Example: "the effect is about 2, and
values from roughly 0 to 4 are all consistent with our data."

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/ci_mechanics.png" width="70%"/></center>

*Build an interval from each of 20 fresh samples and about 19 of the 20 contain the true value; one misses. Your one real interval either contains the truth or it does not. The "95%" is a property of the procedure across many samples, never a probability attached to your single interval. (From Professor Moreira's QM 67000 Business Analytics slides.)*

### 🔍 Reading the Evidence

Look at the two piles and the summary table, then answer in the cell below.
First: the **spread of the 1,000 estimates** and the **average reported SE**
came out nearly equal at each sample size. Why is that the good news that makes
confidence intervals honest? Second: your prediction said the spread would be
cut to ¼, cut to ½, or stay the same. Which was right, and what does the answer
tell you about the *price* of precision? If halving your uncertainty costs 4×
the data, what does cutting it to a quarter cost? Third: a classmate reports
"the effect is 2.04." Using these piles, write the one sentence you would add
to make that claim honest.

### YOUR ANSWER HERE:

**Why spread-of-estimates ≈ average-reported-SE is good news:**

**Which prediction was right, and the price of precision:**

**The sentence that makes "the effect is 2.04" honest:**

---

Precision, in other words, has a price, and the √n rule sets it. You just
watched 4× the data buy only half the uncertainty:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/moe_vs_n.png" width="70%"/></center>

*The margin of error keeps shrinking as the sample grows, but ever more slowly; halving it costs about four times the sample. This is exactly why an honest abstract states its uncertainty instead of pretending precision is free. (From Professor Moreira's QM 67000 Business Analytics slides.)*

## 2. Reach: Who Does Your Answer Actually Speak For?

**Guiding question:** *my estimate wobbles this much, fine. But to whom does it
even apply?*

An interval says how much your number could wobble. It says nothing about
**who the number speaks for**. That second question is the reach half of
generalization, and it is bought or lost by your data strategy long before any
analysis runs. The compass calls the unlicensed version of this move **the
silent upgrade**: a sample narrated as its population with nothing to license
the crossing. Two threats produce it. **Selection** is about who got into your
data in the first place. **Non-response** is about who, once selected, actually
answered. Neither one is noise. Both are tilts, and a tilt does not average
away.

We will make non-response bite a real dataset, measure exactly how far it moves
the answer, and then choose what an honest write-up does about it.

> **A question that often comes up here:** *"If my sample is big, doesn't size
> make up for who is missing?"* No, and this is the most expensive
> misunderstanding in survey research. A bigger sample of the same tilted kind
> just gives you a more precise wrong answer. Ten thousand respondents who all
> lean one way land confidently in the wrong place; four hundred drawn fairly
> can land near the truth. Reach is bought by *how* you sample, never by *how
> many* you sample.

---

### 🔮 Pause & Predict

The LAPOP Brazil survey asked \~10,000 people to rate their trust in the police
on a 1–7 scale; the true average across everyone is about 4.4. Now imagine a
flawed survey where the **least-trusting fifth of people never pick up the
phone**. They are exactly the ones most alienated from institutions, so they
opt out. We will drop them and recompute the average from who is left.
**Before running**, commit: will the observed average move **up**, **down**, or
**stay put**, and roughly how far off the true 4.4 will it land?

### YOUR ANSWER HERE:

**The observed mean will move (up / down / stay):**

**My guess for the observed mean (true value is \~4.4):**

**Why:**

---

> 💡 **Gemini Prompt:** "This code takes a survey where the true average trust in police is about 4.4 on a 1–7 scale, then drops the least-trusting fifth of people (as if they refused to answer) and recomputes the average from who is left: [paste the next cell]. Explain why removing the bottom of the scale pushes the observed average in a predictable direction, and why collecting more respondents of the same kind would NOT fix the problem."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the printed "Non-response bias": does its sign match the direction Gemini predicted (dropping the least-trusting should push the mean up)?
> - [ ] Confirm the observed mean the cell prints sits above the true 4.4, and that this gap is a fixed tilt, not random noise a bigger sample would average away.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Load the survey and measure what non-response would do to a real estimate.
lapop = load_course_data("lapop_brazil.csv")
assert lapop.shape == (10000, 10), "unexpected shape — flag this!"
print("✓ Loaded lapop_brazil.csv —", lapop.shape[0], "rows,", lapop.shape[1], "columns")

v = "trust_police"
true_mean = lapop[v].mean()

# The least-trusting fifth don't respond: drop everyone at or below the 20th
# percentile of trust, then recompute the mean from the respondents who remain.
cutoff = lapop[v].quantile(0.20)
respondents = lapop[lapop[v] > cutoff]
observed_mean = respondents[v].mean()
bias = observed_mean - true_mean

print(f"\nTrue mean trust (everyone):          {true_mean:.2f}")
print(f"Observed mean (least-trusting gone): {observed_mean:.2f}")
print(f"Kept {len(respondents):,} of {len(lapop):,} respondents "
      f"({100*len(respondents)/len(lapop):.0f}%)")
print(f"Non-response bias:                   {bias:+.2f} points")

# Self-check: dropping the LOW end must push the observed mean UP.
assert bias > 0, "dropping the least-trusting should raise the observed mean"
print("\n✓ Self-check passed: silence at the bottom looks like trust at the top.")

### ⚖️ Make a Design Choice

Your survey came back with this exact problem: the least-trusting fifth did not
answer, and your observed mean is biased upward by roughly 0.8 points. You have
three options for what to do next. In the cell below, **pick one and defend it
in a short paragraph**. The defense matters more than the pick.

- **Option A:** Report the observed mean as is, noting non-response in a footnote.
- **Option B:** Report it, but state the direction and rough size of the bias in
  the main text, and bound your claim to "people who respond to surveys like this."
- **Option C:** Do not report a population mean at all; report only the
  association you can defend and call the generalization out of reach.

Name which population each option lets you honestly speak about, and which one
you would want to read if you were Reviewer 2.

### YOUR ANSWER HERE:

**My choice (A / B / C):**

**My defense (and the population it lets me speak about):**

---

### 📝 Practice: the uncertainty-language workshop

Your URC abstract meets its internal gate at this week's Friday studio, and the
fastest way to fail a reviewer is to state a finding as a certainty. Below are
three overcertain sentences. In the cell that follows, rewrite each one so it
**carries its uncertainty without burying the finding**. The result should
still be readable and confident, just honest.

- **A.** "The mentoring program raised belonging by 2 points."
- **B.** "Participants in the program were happier — the data prove it."
- **C.** "Our study found no effect, so mentoring does not work."

> 💡 **Gemini Prompt:** "Here is my draft URC abstract:
> [paste it]. Point out every sentence that states a finding as a certainty, and
> rewrite each to carry uncertainty (a range, an interval, or a hedge) without
> making the abstract vague or burying the main result."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check each rewrite against your ACTUAL numbers. The AI does not know your
>   interval, so confirm any range it invents matches what your simulation produced.
> - [ ] Make sure it did not over-hedge the finding into mush ("may possibly
>   perhaps suggest"); uncertainty is a number, not a shrug.
> - [ ] Log this use in your AI ledger: tool, task, verification.

### YOUR ANSWER HERE:

**A →**

**B →**

**C →**

---

### 🎯 Project Transfer — your headline, wearing its uncertainty

In the cell below, write your project's **headline finding-to-be**, the one
sentence a URC visitor would remember. Then rewrite it to carry uncertainty in
**≤ 25 words**. You will not have real numbers yet; use a placeholder interval
(for example, "give or take X") so the *shape* of the honest sentence is locked
in before the data arrive. This sentence goes straight into the abstract you
polish at this week's studio.

### YOUR ANSWER HERE:

**My headline finding-to-be (first draft):**

**Same headline, ≤ 25 words, carrying its uncertainty:**

---

### 🎟️ Claim Ticket

**Claim Ticket #1** — your headline finding-to-be, stated **with** its
uncertainty and naming who it honestly speaks for, in ≤ 25 words. This line is
the spine of the abstract you bring to this week's studio.

### YOUR ANSWER HERE:

**My headline + uncertainty, ≤ 25 words:**

---

---

# Lecture 2

## 3. Power: Could Your Design Even Find It?

> *"Before you tell me what you found, tell me whether you *could* have found it.
> A study that had a 15% chance of detecting a real effect and then reports 'no
> significant effect' hasn't learned that the effect is absent. It has learned
> that its own eyes were closed. I want the power calculation you ran *before* you
> collected a single data point."*
> — a **skeptical peer** at your practice talk, asking the question that ends careers

Notice something uncomfortable about Lecture 1's very first simulation. With
n = 100, the true effect is a real 2 points, and yet the 95% interval routinely
stretched from below zero to above four. A study like that will fail to "find"
its own real effect most of the time. Flip that failure rate around and you get
**statistical power**: the probability your design detects a real effect of a
given size. Example: power of 15% means that even when the effect is truly
there, about five studies out of six built like yours would still report
"nothing found." Low power is why "we found nothing" is so often a statement
about the *design*, not the world.

Power lives on a triangle. You can raise it three ways:

- **More people (n):** the reliable lever, but precision only grows with √n.
- **Less noise:** cleaner measurement and tighter designs; every point of noise
  you remove sharpens the eyes.
- **A bigger effect:** usually not yours to choose, though a stronger "dose" of
  an intervention sometimes is.

We will simulate power across a grid of n and effect size, find where the
mentoring design actually sits, and locate the change that would raise it most.
This is exactly the reasoning your one-page M07 declaration has to survive at
this week's studio.

> **A question that often comes up here:** *"Is it unethical to run an underpowered
> study?"* It depends on what you claim from it. Running a small pilot and
> reporting it honestly as "underpowered, suggestive, here's the interval" is
> good science. Running the same study and announcing "no effect" is the sin: you
> spent people's time to manufacture a false certainty. Power is what tells you
> which sentence you are entitled to.

---

### 🔮 Pause & Predict

We will compute power across a grid: sample sizes n ∈ {50, 100, 200, 400}
crossed with true effect sizes {0.5, 1, 2, 4} points, running the mentoring
design 1,000 times in each of the 16 cells. **Before running**, commit two
predictions in the cell below. (1) At the **smallest** combination (n = 50,
effect = 0.5), roughly what fraction of studies will detect the effect? (2) At
the **largest** (n = 400, effect = 4), roughly what fraction? Give two
percentages.

### YOUR ANSWER HERE:

**Power at n = 50, effect = 0.5 (my guess, %):**

**Power at n = 400, effect = 4 (my guess, %):**

---

> 💡 **Gemini Prompt:** "In this code, statistical power is the fraction of 1,000 simulated studies whose 95% interval excludes zero, computed across a grid of sample sizes crossed with true effect sizes: [paste the next cell]. Explain in plain words what 'power' means for a real study, and predict which corner of the grid will show the highest power and which the lowest: small n with a small effect, or large n with a large effect."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Read the printed grid: does the largest-n / largest-effect cell really show the highest power, and the smallest / smallest the lowest, as Gemini predicted?
> - [ ] Scan the smallest-effect row and confirm that adding people barely moves power there, then be ready to say why a tiny effect stays hard to detect at any n in this range.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Power = the fraction of 1,000 simulated studies whose 95% interval excludes 0
# (equivalently, |estimate / SE| > 1.96). We reuse the SAME run_design engine.
def power_at(n, effect, noise=2.0, reps=1000, seed=SEED):
    d = run_design(n=n, effect=effect, noise=noise, reps=reps, seed=seed)
    return (np.abs(d.est / d.se) > 1.96).mean()

ns = [50, 100, 200, 400]
effects = [0.5, 1.0, 2.0, 4.0]
grid = pd.DataFrame(
    {e: [power_at(n, e) for n in ns] for e in effects},
    index=[f"n={n}" for n in ns],
)
grid.columns = [f"effect={e}" for e in effects]
print("Statistical power (share of 1,000 studies that detect the effect):")
print(grid.round(3).to_string())

# Self-check: the biggest design/effect must out-detect the smallest.
strongest = grid.loc["n=400", "effect=4.0"]
weakest   = grid.loc["n=50",  "effect=0.5"]
assert strongest > weakest, "power should rise as n and effect grow"
print(f"\n✓ Self-check passed: power climbs from {weakest:.0%} (smallest) "
      f"to {strongest:.0%} (largest).")

The same lesson, drawn by the textbook across three sample sizes at once:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_11_2.png" width="70%"/></center>

*Power climbs with the true effect size, and a bigger sample lifts the whole curve: at N = 1000 even fairly small effects clear the conventional 0.8 power target, while at N = 100 only large effects do. (Figure from the replication materials of Blair, Coppock & Humphreys (2023), *Research Design in the Social Sciences* (MIT-licensed archive; the book is free at book.declaredesign.org).)*

### 🔍 Reading the Evidence

Using the grid, answer in the cell below. First: the mentoring design in this
course sits at roughly **n = 100, effect = 2**. Read its power off the grid,
and say in plain words what that percentage means for a real study. Second:
find the sample size that would roughly double that power (scan the
`effect=2.0` column). How many more people is that? Third: look across the
`effect=0.5` row. Why does adding people barely help when the true effect is
tiny, and what does that teach you about designs chasing very small effects?

### YOUR ANSWER HERE:

**Power of the n=100, effect=2 design, in plain words:**

**The n that would roughly double it, and how many more people that is:**

**Why n barely helps when the effect is tiny:**

---

### 📝 Practice: rank the fixes, then verify

Your mentoring design sits at n = 100, effect = 2, noise = 2, with power \~15%.
A grant will fund exactly **one** upgrade. Rank these three by how much each
raises power, then check your ranking against the grid and the helper.

- **A.** Double the sample: n = 100 → 200.
- **B.** Recruit for a stronger program so the effect is 4 instead of 2.
- **C.** Keep n = 100 and the effect at 2, but tighten measurement so noise
  drops from 2 to 0.

Predict the order first, then run the verification cell.

> 💡 **Gemini Prompt:** "For a two-group study whose power is only about 15%, I can make exactly one upgrade: double the sample, raise the true effect from 2 to 4, or cut the extra measurement noise to zero. Rank these three by how much each should raise statistical power, explain why a bigger effect size usually helps most, and explain why scrubbing a small noise term helps least when the outcome is already naturally very spread out."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check Gemini's ranking against the three printed power numbers (A double n, B bigger effect, C less noise): does its predicted order match what the simulation actually shows?
> - [ ] If it claims cutting noise is a big win, confirm against the tiny gain the cell prints; here the outcome's natural spread dominates the small noise term.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# YOUR SOLUTION HERE — read these three numbers and rank the fixes by power gain.
base = power_at(100, effect=2.0, noise=2.0)
fix_A = power_at(200, effect=2.0, noise=2.0)   # double n
fix_B = power_at(100, effect=4.0, noise=2.0)   # bigger effect
fix_C = power_at(100, effect=2.0, noise=0.0)   # less measurement noise
print(f"Baseline (n=100, effect=2, noise=2): power = {base:.0%}")
print(f"A  double n         -> {fix_A:.0%}")
print(f"B  bigger effect    -> {fix_B:.0%}")
print(f"C  less noise       -> {fix_C:.0%}")

### YOUR ANSWER HERE:

**My predicted ranking (most power gained first):**

**The ranking the simulation showed:**

**What surprised me:**

---

## 4. The Boundary: Association Is Not Causation

**Guiding question:** *what am I allowed to say about **why**?*

> *"I read a lot of abstracts. The trick I catch most often isn't a bad number.
> It's a good number that grows a leg. Paragraph one says 'associated.' Paragraph
> three says 'linked to.' The conclusion says 'impact.' Nobody added evidence
> between paragraphs one and three. They just added a stronger verb."*
> — a **journal reviewer**, describing the most common way a paper overclaims

One boundary is left, and it is the one your abstract is most likely to cross
by accident. An **association** means two things move together in your data.
Example: people who use the campus tutoring center also tend to have higher
GPAs. **Causation** means one thing actually moves the other: tutoring *raises*
GPA. Data alone can show the first. Only a design with the right machinery can
license the second: random assignment, or the kind of identification argument
nb13 teaches (a reasoned case that your comparison isolates the cause).

The crossing rarely happens in one honest step. It happens rung by rung on a
**language ladder**, each rung a stronger claim your design never bought:

| Rung | Sounds like | What it needs |
|---|---|---|
| **associated** | "X and Y move together in my data" | a clean measurement (you may be here) |
| **linked to** | "X is linked to Y" | the same thing, dressed up; still just association |
| **predicts** | "X predicts Y" | out-of-sample forecasting (nb12) |
| **impacts / affects** | "X impacts Y" | an identification argument (nb13) |
| **causes** | "X causes Y" | a randomized or credibly-identified design |

> **A question that often comes up here:** *"If I can't say 'causes,' hasn't my
> whole project failed?"* Not at all. A precise, honestly-bounded association is
> a genuine contribution, and most published quantitative findings live exactly
> there. The failure is not stopping at "associated". The failure is *starting*
> at "associated" and *ending* at "causes" while pretending nothing changed.

*(If you want to see the ladder in the wild, the "correlation ≠ causation" case
index at [callingbullshit.org/case_studies.html](https://callingbullshit.org/case_studies.html)
collects real examples of this exact slip.)*

---

### 🎯 Project Transfer — your declaration's honest spine (M07)

Your full one-page M07 declaration is assembled at this week's studio. In the
cell below, write two of its lines now. **The power line:** your design's
target sample size, the smallest effect you would care to detect, and, reasoning
from today's grid, a rough power estimate for that combination, plus the single
change you would make first to raise it. If your real design is more complex
than the two-group world, borrow the closest cell of the grid and say so; the
*reasoning* is what M07 is graded on, not a decimal. **The claim line:** the
one sentence stating what your analysis will find, naming the exact population
your data can reach and sitting on the rung your design actually buys, with the
one word you are most tempted to smuggle upward replaced by its honest
alternative.

### YOUR ANSWER HERE:

**Target n:** &nbsp; **Smallest effect worth detecting:** &nbsp; **Rough power:**

**The one change I would make first to raise my power:**

**My claim line (population named, correct rung; tempting word → honest replacement):**

---

### 🎟️ Claim Ticket

**Claim Ticket #2** — your design's approximate power (a percentage) and the
single change that would raise it most. This line closes the generalization
deep dive and goes into the M07 declaration you assemble at this week's
studio.

### YOUR ANSWER HERE:

**My approximate power + the single change that would raise it most:**

---

## 5. If You Want to Go Further *(optional)*

Nothing below is required, and nothing at this week's studios depends on it. It
is here for the moment your own project makes you curious.

**The full power curve.** The grid sampled four values of n. The cell below
traces the whole curve for our real 2-point effect, showing how much n it takes
to reach the conventional 80% power target.

In [ ]:
# OPTIONAL — the power curve for OUR mentoring effect (a real 2 points) as n grows.
curve_ns = [50, 100, 200, 400, 800]
curve = [power_at(n, effect=2.0) for n in curve_ns]

fig, ax = plt.subplots()
ax.plot(curve_ns, curve, marker="o", color="#268")
ax.axhline(0.80, color="k", linestyle="--", label="80% power (a common target)")
ax.set_xlabel("sample size n")
ax.set_ylabel("power (chance of detecting the real 2-point effect)")
ax.set_title("How much n it takes to reliably see a real effect")
ax.set_ylim(0, 1)
ax.legend()
plt.show()

for n, p in zip(curve_ns, curve):
    print(f"  n={n:>3}: power = {p:.0%}")

### 📝 Practice: climb down the ladder *(optional extra drill)*

Each sentence below sits higher on the language ladder than its evidence can
support. In the cell that follows, rewrite each one **down to the rung the
stated design actually bought**, and name the rung you landed on.

- **A.** "Our survey shows that using the campus tutoring center **causes** higher
  GPAs." *(evidence: a one-time survey; tutoring use and GPA were both measured at
  the same moment.)*
- **B.** "Screen time **impacts** teen sleep." *(evidence: teens who report more
  screen time also report less sleep, in one questionnaire.)*
- **C.** "This poll of our club members shows most **undergraduates** support the
  proposal." *(evidence: 40 club members, all volunteers, answered.)*

### YOUR ANSWER HERE:

**A → (rewrite + rung):**

**B → (rewrite + rung):**

**C → (rewrite + rung):**

---

## 6. Wrap-Up

Across two lectures you turned "here is my number" into "here is my number,
here is how much it could wobble, here is who it speaks for, and here is
whether I could even have seen it." You built a **sampling distribution** by
brute-force repetition and read the **standard error** off its spread. You
watched the **√n rule** charge you 4× the data for half the uncertainty. You
made **non-response** tilt a real survey and measured the exact lean. You
computed **power** before collecting a thing, so that "we found nothing" can
never again sneak out of a design whose eyes were closed. And you learned to
keep your verbs on the rung your design paid for.

> **"An estimate without its uncertainty is a rumor; a null result without its
> power is a shrug dressed as a finding. Report both, always."**

Next, nb11 takes the very same mentoring world and hands you the course's
**diagnosis engine**: a way to *declare* a design in code and *diagnose* its
bias, power, and coverage before you ever run it, then *redesign* it to fix
what the diagnosis exposes. Everything you simulated by hand here becomes a
single reusable tool. Bring your M07 declaration; nb11 is where you learn to
stress-test it. This serves milestone M08, your Design Diagnosis & Redesign
Memo.

---

## 7. Sources & Provenance

**Provenance of this notebook:**
- *RDSS ch. 9 'Choosing an answer strategy' + ch. 10 'Diagnosing designs' | uncertainty via simulation | sampling-distribution + standard-error simulation built on the source concept | newly-constructed-from-source-concept*
- *RDSS ch. 15 'Observational: descriptive' + ch. 16 'Observational: causal' | the association→causation boundary | language-ladder + non-response stress-test built on the concept | extended*
- *rdss-problem-set-1 ('Introduction to power analysis with design diagnosis') + RDSS ch. 10 §power | power by simulation | power-grid translated R→Python and recast as a course simulation | translated (R→Python)*
- *lapop_brazil.csv | rdss package data | trust-in-institutions items | adapted (teaching resample; non-response injected for the generalization lesson)*
- *nb04 mentoring-program world (make_world) | course-internal | potential-outcomes world reused as this notebook's simulation sandbox | reused*

**Dataset attribution:** Dataset from the `rdss` package (Blair, Coppock &
Humphreys, MIT License), companion to *Research Design in the Social Sciences*
(2023).

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the Social
  Sciences*, ch. 9–10 and ch. 15–16. Free online:
  [book.declaredesign.org](https://book.declaredesign.org/).
- *Optional:* the "correlation ≠ causation" case index at
  [callingbullshit.org/case_studies.html](https://callingbullshit.org/case_studies.html).

---

<center>

Thank you!

</center>